In [ ]:
import subprocess
import time
import shutil
import os
import re
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed


# ---------------------------------------------------------------------------
# OneDrive helpers
# ---------------------------------------------------------------------------

def force_onedrive_download(path):
    """Force OneDrive/SharePoint to download a cloud-only file."""
    try:
        subprocess.run(["attrib", "-U", path], check=True, shell=True)
    except Exception as e:
        print(f"⚠️ Failed forcing OneDrive download for {path}: {e}")


def set_onedrive_online_only(path):
    """Mark a file as online-only in OneDrive/SharePoint."""
    try:
        subprocess.run(["attrib", "+U", path], check=True, shell=True)
    except Exception as e:
        print(f"⚠️ Failed setting online-only for {path}: {e}")


# ---------------------------------------------------------------------------
# Safe copy
# ---------------------------------------------------------------------------

def safe_copy(src, dst, retries=8, delay=8):
    """Attempt to copy a file, retrying if OneDrive cloud timeout happens."""
    for attempt in range(1, retries + 1):

        try:
            shutil.copy2(src, dst)

            # Free local cache immediately after successful copy
            set_onedrive_online_only(src)

            return True

        except OSError as e:

            msg = str(e)

            if (
                "WinError 426" in msg
                or "cloud operation" in msg
                or "0x80070185" in msg
            ):
                print(
                    f"⏳ Waiting for file to sync: "
                    f"{os.path.basename(src)} "
                    f"(attempt {attempt}/{retries})"
                )

                force_onedrive_download(src)
                time.sleep(delay)
                continue

            print(f"❌ Error copying {src}: {e}")
            return False

    print(f"❌ Timeout after {retries} attempts for {src}")
    return False


# ---------------------------------------------------------------------------
# Extraction
# ---------------------------------------------------------------------------

def extract_selected_data(
    csv_path,
    source_root,
    destination_root,
    modalities_to_extract=None,
    filename_patterns=None,
    max_workers=5,
):
    """
    Copy selected participants' data (anat, func, dwi)
    preserving dataset structure.

    max_workers controls the maximum number of simultaneous
    OneDrive downloads/copies.
    """

    df = pd.read_csv(
        csv_path,
        delimiter=",",
        converters={"session": lambda x: str(x)}
    )

    ###########################################################################
    # Build copy jobs
    ###########################################################################

    copy_jobs = []

    for _, row in df.iterrows():

        dataset = row["dataset"]
        participant = row["participant_id"]
        session = str(row["session"])

        if modalities_to_extract is None:
            modalities = {
                "anat": bool(row.get("anat", 0)),
                "func": bool(row.get("fmri", 0)),
                "dwi": bool(row.get("dwi", 0)),
            }
        else:
            valid_modalities = {
                "anat": "anat",
                "fmri": "func",
                "dwi": "dwi",
            }
            modalities = {
                valid_modalities[m]: True
                for m in modalities_to_extract
                if m in valid_modalities
            }

        dataset_path = os.path.join(source_root, dataset)
        data_path = os.path.join(dataset_path, "data")

        #######################################################################
        # Dataset-level files
        #######################################################################

        dest_dataset_root = os.path.join(destination_root, dataset)
        os.makedirs(dest_dataset_root, exist_ok=True)

        if not os.path.isdir(dataset_path):
            print(f"⚠️ Missing dataset folder: {dataset_path}")
            continue

        for item in os.listdir(dataset_path):

            src_item = os.path.join(dataset_path, item)

            if os.path.isdir(src_item):
                continue

            dst_item = os.path.join(dest_dataset_root, item)

            copy_jobs.append(
                (
                    src_item,
                    dst_item,
                    f"dataset file {dataset}/{item}",
                )
            )

        #######################################################################
        # Subject/session validation
        #######################################################################

        subj_path = os.path.join(data_path, f"sub-{participant}")

        if not os.path.isdir(subj_path):
            print(f"⚠️ Missing subject folder: {subj_path}")
            continue

        ses_path = (
            os.path.join(subj_path, f"ses-{session}")
            if session != "NA"
            else subj_path
        )

        if not os.path.isdir(ses_path):
            print(f"⚠️ Missing session folder: {ses_path}")
            continue

        dest_ses_path = os.path.join(
            dest_dataset_root,
            "data",
            f"sub-{participant}",
        )

        if session != "NA":
            dest_ses_path = os.path.join(
                dest_ses_path,
                f"ses-{session}",
            )

        os.makedirs(dest_ses_path, exist_ok=True)

        #######################################################################
        # Modalities
        #######################################################################

        for modality, flag in modalities.items():

            if not flag:
                continue

            src_modality_path = os.path.join(
                ses_path,
                modality,
            )

            if not os.path.isdir(src_modality_path):
                print(
                    f"⚠️ {modality} folder not found: "
                    f"{src_modality_path}"
                )
                continue

            dest_modality_path = os.path.join(
                dest_ses_path,
                modality,
            )

            os.makedirs(dest_modality_path, exist_ok=True)

            ###################################################################
            # Copy everything
            ###################################################################

            if filename_patterns is None:

                for root, _, files in os.walk(src_modality_path):

                    rel_path = os.path.relpath(
                        root,
                        src_modality_path,
                    )

                    dest_dir = os.path.join(
                        dest_modality_path,
                        rel_path,
                    )

                    os.makedirs(dest_dir, exist_ok=True)

                    for fname in files:

                        copy_jobs.append(
                            (
                                os.path.join(root, fname),
                                os.path.join(dest_dir, fname),
                                f"{dataset}/{participant}/{session}/{modality}",
                            )
                        )

                continue

            ###################################################################
            # Filtered copy
            ###################################################################

            for root, _, files in os.walk(src_modality_path):

                for fname in files:

                    if not any(
                        re.search(pattern, fname)
                        for pattern in filename_patterns
                    ):
                        continue

                    rel_path = os.path.relpath(
                        root,
                        src_modality_path,
                    )

                    dest_dir = os.path.join(
                        dest_modality_path,
                        rel_path,
                    )

                    os.makedirs(dest_dir, exist_ok=True)

                    copy_jobs.append(
                        (
                            os.path.join(root, fname),
                            os.path.join(dest_dir, fname),
                            f"{dataset}/{participant}/{session}/{modality}",
                        )
                    )

    ###########################################################################
    # Parallel copy
    ###########################################################################

    print(
        f"\nCopying {len(copy_jobs)} files "
        f"using {max_workers} workers...\n"
    )

    def worker(job):

        src, dst, label = job

        ok = safe_copy(src, dst, retries=8, delay=10)

        return ok, label

    copied = 0
    failed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:

        futures = [executor.submit(worker, job) for job in copy_jobs]

        for future in as_completed(futures):

            ok, label = future.result()

            if ok:
                copied += 1
            else:
                failed += 1
                print(f"⚠️ Failed: {label}")

    print("\nExtraction complete.")
    print(f"Copied: {copied}")
    print(f"Failed: {failed}")


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------

print("Patients")

extract_selected_data(
    csv_path="../Analyses/FranA9/Patients.csv",
    source_root="../Storage Repository/",
    destination_root="../Analyses/FranA9/datasets_patients/",
    modalities_to_extract=["anat"],
    filename_patterns=[
        r"^cat_sub-.*_T1w\.xml$",
        r"^catROI_sub-.*_T1w\.xml$",
    ],
    max_workers=3,
)

print("HC")

extract_selected_data(
    csv_path="../Analyses/FranA9/HC.csv",
    source_root="../Storage Repository/",
    destination_root="../Analyses/FranA9/datasets_hc/",
    modalities_to_extract=["anat"],
    filename_patterns=[
        r"^cat_sub-.*_T1w\.xml$",
        r"^catROI_sub-.*_T1w\.xml$",
    ],
    max_workers=3,
)

Patients
⚠️ anat folder not found: ../Storage Repository/COMSS\data\sub-COMSS101\ses-A\anat
⚠️ anat folder not found: ../Storage Repository/COMSS\data\sub-COMSS102\ses-A\anat
⚠️ anat folder not found: ../Storage Repository/COMSS\data\sub-COMSS103\ses-A\anat
⚠️ anat folder not found: ../Storage Repository/COMSS\data\sub-COMSS104\ses-A\anat
⚠️ anat folder not found: ../Storage Repository/COMSS\data\sub-COMSS105\ses-A\anat
⚠️ anat folder not found: ../Storage Repository/COMSS\data\sub-COMSS106\ses-A\anat
⚠️ anat folder not found: ../Storage Repository/COMSS\data\sub-COMSS107\ses-A\anat
⚠️ anat folder not found: ../Storage Repository/COMSS\data\sub-COMSS108\ses-A\anat
⚠️ anat folder not found: ../Storage Repository/COMSS\data\sub-COMSS109\ses-A\anat
⚠️ anat folder not found: ../Storage Repository/COMSS\data\sub-COMSS110\ses-A\anat
⚠️ anat folder not found: ../Storage Repository/COMSS\data\sub-COMSS111\ses-A\anat
⚠️ anat folder not found: ../Storage Repository/COMSS\data\sub-COMSS112\ses-A\